# 04 - Testing the grid tracker

Week 3. The tracker in `src/tracker.py` measures how far the ground moved between
two images. Before trusting a single number it produces, it has to be checked
against cases where the answer is already known.

Four tests, cheapest first. Stop at the first one that fails - the later tests
assume the earlier ones passed.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

from src.testbed import move_patch
from src.tracker import track_grid

DATA = "../data/chamoli"
settings = pd.read_csv(f"{DATA}/settings.csv", header=None, index_col=0).squeeze("columns")

raw = np.load(f"{DATA}/{settings['base_date']}.npy").astype("float64")
base = (raw - raw.min()) / (raw.max() - raw.min())

# The block that moves. settings.csv holds the scar centre; the box there is the
# darkest core of the change (about 0.04 km2), too small for a 32 px window to sit
# inside. 50 px square matches the published 0.25 km2 of rock and ice that fell.
SCAR_ROW = int(settings["scar_row"])
SCAR_COL = int(settings["scar_col"])
SIDE = 50
half = SIDE // 2
BOX = (SCAR_ROW - half, SCAR_ROW + half, SCAR_COL - half, SCAR_COL + half)

print(f"base image: {settings['base_date']}  {base.shape}")
print(f"block: rows {BOX[0]}-{BOX[1]}, cols {BOX[2]}-{BOX[3]}")


def inside_block(result):
    """The grid cells whose centres fall within the moving block."""
    rsel = (result["rows"] >= BOX[0]) & (result["rows"] < BOX[1])
    csel = (result["cols"] >= BOX[2]) & (result["cols"] < BOX[3])
    return np.ix_(rsel, csel)

## Test 1: the same image, twice

Nothing moved, so everything must read exactly zero. If this fails, stop - no
later test is worth running.

In [ ]:
r = track_grid(base, base, window=32, step=16)
print(f"max |dx| = {np.nanmax(np.abs(r['dx'])):.2e}")
print(f"max |dy| = {np.nanmax(np.abs(r['dy'])):.2e}")
print(f"{np.isfinite(r['dx']).sum()} of {r['dx'].size} windows measured")

Two blank maps. Same colour scale as test 2, so the contrast between nothing-moved and something-moved is immediate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

lo, hi = np.percentile(base, [2, 98])
axes[0].imshow(np.clip((base - lo) / (hi - lo), 0, 1), cmap="gray")
axes[0].set_title("the image, compared with itself")

for ax, key in zip(axes[1:], ["dx", "dy"]):
    im = ax.imshow(r[key], cmap="RdBu", vmin=-2.5, vmax=2.5)
    ax.set_title(f"{key}: every cell reads 0")
    plt.colorbar(im, ax=ax, fraction=0.046)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Test 2: a known shift

Move the block by exactly 2 pixels, then measure it back.

Two things matter here, and the second one matters more:

- **Inside the block** should read close to 2.0.
- **Outside the block** must read zero. Nothing out there moved, so anything the
  tracker reports is motion it invented.

In [ ]:
TRUE_SHIFT = 2.0
before, after = move_patch(base, TRUE_SHIFT, 0.0, BOX)
r = track_grid(before, after, window=32, step=8)

idx = inside_block(r)
outside = np.ones(r["dx"].shape, bool)
outside[idx] = False

print(f"imposed:  {TRUE_SHIFT:+.3f}")
print(f"inside:   {np.nanmedian(r['dx'][idx]):+.3f}")
print(f"outside:  {np.nanmedian(r['dx'][outside]):+.3f}   (must be 0)")
print(f"error:    {np.nanmedian(r['dx'][idx]) - TRUE_SHIFT:+.3f}")

Before and after look identical - a 2 px shift on a 110 px crop is nearly invisible, which is the whole premise of the project.

The third panel is the one that matters: it lights up **only inside the red square** and stays flat everywhere else. If colour appeared outside it, `move_patch` would be shifting more than it should, and every measurement after this would be wrong.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
PAD = 30

for ax, img, title in [(axes[0], before, "before"), (axes[1], after, "after")]:
    sub = img[BOX[0]-PAD:BOX[1]+PAD, BOX[2]-PAD:BOX[3]+PAD]
    lo, hi = np.percentile(sub, [2, 98])
    ax.imshow(np.clip((sub - lo) / (hi - lo), 0, 1), cmap="gray")
    ax.add_patch(Rectangle((PAD, PAD), SIDE, SIDE, fill=False, ec="#ff3b30", lw=2))
    ax.set_title(title)

d = (after - before)[BOX[0]-PAD:BOX[1]+PAD, BOX[2]-PAD:BOX[3]+PAD]
im = axes[2].imshow(d, cmap="RdBu", vmin=-0.08, vmax=0.08)
axes[2].add_patch(Rectangle((PAD, PAD), SIDE, SIDE, fill=False, ec="#ff3b30", lw=2))
axes[2].set_title("after \u2212 before: only the block changed")
plt.colorbar(im, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Test 3: look at the map

The numbers can pass while the geometry is wrong. One coloured square sitting
where the block is, flat everywhere else.

If the square is in the wrong place, or the whole map is coloured, there is an
indexing bug in the loop.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(r["dx"], cmap="RdBu", vmin=-2.5, vmax=2.5)
axes[0].set_title("dx: east-west")
plt.colorbar(im, ax=axes[0], fraction=0.046)

im = axes[1].imshow(r["dy"], cmap="RdBu", vmin=-2.5, vmax=2.5)
axes[1].set_title("dy: north-south (nothing imposed)")
plt.colorbar(im, ax=axes[1], fraction=0.046)

for ax in axes:
    ax.set_xlabel("grid column")
    ax.set_ylabel("grid row")
plt.tight_layout()
plt.show()

## Test 4: does the window size matter?

The one real decision inside the tracker. A bigger window has more texture to
match against, so the answer should be steadier - but it averages over more
ground, so where it straddles the block's edge it mixes moving ground with
stable ground.

The block is 50 px across. A 64 px window cannot fit inside it; a 16 px one
has little to work with.

Two things are measured for each size: how close the reading is to 2.0, and how
sharp the block's edge looks in the map.

**Use `step=8`.** At step 16 only three grid cells fall across a 50 px block,
which is too coarse to see an edge at all.

In [ ]:
STEP = 8
rows, profiles = [], {}

for win in [16, 32, 64]:
    rr = track_grid(before, after, window=win, step=STEP)

    reading = float(np.nanmedian(rr["dx"][inside_block(rr)]))

    # one horizontal line of the map, straight through the block's centre
    i = int(np.argmin(np.abs(rr["rows"] - SCAR_ROW)))
    prof = rr["dx"][i]
    profiles[win] = (rr["cols"], prof)

    # crude edge width: grid cells sitting at an in-between value
    edge = int(np.sum(np.isfinite(prof)
                      & (np.abs(prof) > 0.25 * TRUE_SHIFT)
                      & (np.abs(prof) < 0.75 * TRUE_SHIFT)))

    rows.append({
        "window": win,
        "grid": f"{rr['dx'].shape[0]}x{rr['dx'].shape[1]}",
        "reading": round(reading, 3),
        "error": round(reading - TRUE_SHIFT, 3),
        "edge_cells": edge,
    })

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for win, (cols, prof) in profiles.items():
    ax.plot(cols, prof, marker="o", ms=3, label=f"window = {win}")

ax.axhline(TRUE_SHIFT, ls="--", lw=1, color="k", label=f"imposed {TRUE_SHIFT} px")
ax.axvspan(BOX[2], BOX[3], alpha=0.12, color="tab:red", label="the moving block")
ax.set_xlim(BOX[2] - 90, BOX[3] + 90)
ax.set_xlabel("image column")
ax.set_ylabel("measured dx (px)")
ax.set_title("a slice through the block centre")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### What to look for

- **The small window wobbles** inside the block, where the others hold steady.
  Too little texture per window to match reliably, so the expected gain in detail
  does not appear.
- **The large window has the widest ramp** at the block's edges, and spills past
  them. A 64 px window straddling the boundary averages moving ground with still
  ground.
- **None of them reach 2.0.** If the shortfall is there at every window size,
  that rules out "the window is too small" as its cause - one of the four
  suspects, eliminated.

Write down which suspect this removes and what is left. An error you have
narrowed down is worth far more than one you have only noticed.

## Test 5: co-registration

The satellite never points at exactly the same spot twice. Sentinel-2 places a
pixel to about 10 m - one whole pixel - while the slope we care about creeps
roughly 0.2 px a year. The pointing error is several times the signal.

`coregister` removes it. The idea is simple: ground that did not move still
reads non-zero, and that reading *is* the misalignment. Measure it over ground
you trust, subtract it from everywhere.

This test imposes a known misalignment on top of a known block shift, then
checks that the function gives the block shift back.

In [ ]:
from src.testbed import misalign
from src.tracker import coregister

OFFSET = 0.6                     # the fake pointing error, in pixels
STABLE = (300, 400, 100, 200)    # a patch of ground far from the block

b5, a5 = move_patch(base, TRUE_SHIFT, 0.0, BOX)
b5, a5 = misalign(b5, a5, OFFSET, 0.0)

r5 = track_grid(b5, a5, window=32, step=8)

idx5 = inside_block(r5)
out5 = np.ones(r5["dx"].shape, bool)
out5[idx5] = False

print(f"imposed block shift:   {TRUE_SHIFT:+.1f}")
print(f"imposed misalignment:  {OFFSET:+.1f}")
print()
print("measured, before co-registration")
print(f"  inside block:   {np.nanmedian(r5['dx'][idx5]):+.3f}   (should sit near {TRUE_SHIFT + OFFSET:+.1f})")
print(f"  outside block:  {np.nanmedian(r5['dx'][out5]):+.3f}   (should sit near {OFFSET:+.1f})")

Nothing is still any more. Ground that never moved reads about +0.53 instead of
0, and the block reads about +2.26 instead of +2.0. The whole map has been
lifted by the pointing error.

This is what every real Sentinel-2 pair looks like, and it is why a raw
displacement map cannot be read directly. A slope creeping 0.2 px a year would
be sitting underneath an error three times its size.

In [ ]:
c5 = coregister(r5, STABLE)

off_dx, off_dy = c5["offset"]
print(f"stable windows used:   {c5['n_stable_windows']}")
print(f"misalignment found:    {off_dx:+.3f}, {off_dy:+.3f}   (imposed {OFFSET:+.1f}, +0.0)")
print()
print("measured, after co-registration")
print(f"  inside block:   {np.nanmedian(c5['dx'][idx5]):+.3f}   (truth {TRUE_SHIFT:+.1f})")
print(f"  outside block:  {np.nanmedian(c5['dx'][out5]):+.3f}   (truth  0.0)")
print(f"  dy inside:      {np.nanmedian(c5['dy'][idx5]):+.3f}   (truth  0.0, nothing moved north-south)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, res, title in [(axes[0], r5, f"before: everything lifted by {OFFSET} px"),
                       (axes[1], c5, "after: only the block is left")]:
    im = ax.imshow(res["dx"], cmap="RdBu", vmin=-2.5, vmax=2.5)
    ax.set_title(title)
    ax.set_xlabel("grid column")
    plt.colorbar(im, ax=ax, fraction=0.046, label="dx (px)")

# outline the stable region, converted from image pixels to grid cells
gr = (r5["rows"] >= STABLE[0]) & (r5["rows"] < STABLE[1])
gc = (r5["cols"] >= STABLE[2]) & (r5["cols"] < STABLE[3])
axes[0].add_patch(Rectangle((np.argmax(gc) - 0.5, np.argmax(gr) - 0.5),
                            gc.sum(), gr.sum(), fill=False, ec="k", lw=1.5, ls="--"))
axes[0].text(np.argmax(gc), np.argmax(gr) - 1.5, "stable region", fontsize=9)

plt.tight_layout()
plt.show()